### Análise Exploratória da Arrecadação do ITR no Brasil

# Contexto do projeto

Este projeto foi desenvolvido na disciplina Python para Engenharia de Dados, durante a Pós-Graduação em Engenharia de Dados.

## Objetivo

Realizar as etapas de ingestão, limpeza, transformação, avaliação da qualidade e análise exploratória dos dados de arrecadação do Imposto sobre a Propriedade Territorial Rural — ITR no Brasil.

A análise busca identificar padrões temporais e geográficos, além de possíveis diferenças na arrecadação entre estados, regiões e municípios.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# Etapa 1 — Ingestão e Transformação dos Dados

Nesta etapa, são realizadas as seguintes atividades:

• Leitura do conjunto de dados;
• Identificação e correção de inconsistências na coluna de valores;
• Remoção de colunas auxiliares;
• Padronização dos nomes das colunas;
• Tratamento dos campos textuais;
• Conversão dos tipos de dados;
• Separação das informações de município e unidade federativa;
• Exportação do conjunto de dados tratado.

In [ ]:
df = pd.read_csv('../data/raw/valores_arrecadacao_itr.csv', sep=';', encoding='latin1', dtype=str)

df.head()

In [ ]:
print(df.columns)

In [ ]:
df.loc[df['Unnamed: 6'].notna()]

In [ ]:
df['Valor'] = df['Valor'].fillna(df['Unnamed: 6'])

df['Valor'] = (
    df['Valor']
    .astype(str)
    .str.replace(',', '.', regex=False)
)

df['Valor'] = pd.to_numeric(df['Valor'], errors='coerce')

df.drop(columns=['Unnamed: 6'], inplace=True)

In [ ]:
for col in df.columns:
    print(f"\nColuna: {col}")
    print(df[col].unique())

In [ ]:
df.columns = [
    'ano',
    'mes',
    'uni_federativa',
    'regiao',
    'cidade_uf',
    'valor'
]
print(df.columns)

df.head()

In [ ]:
colunas_texto = ['mes', 'uni_federativa', 'regiao']

for col in colunas_texto:
    df[col] = (
        df[col].str.strip().str.title()
    )
for col in ['mes', 'uni_federativa', 'regiao']:
    print(f"\nColuna: {col}")
    print(df[col].unique())

In [ ]:
df[['cidade', 'uf_sigla']] = df['cidade_uf'].str.rsplit(' - ', n=1, expand=True)

df['cidade'] = df['cidade'].str.strip()
df['uf_sigla'] = df['uf_sigla'].str.strip()

df.drop(columns=['cidade_uf'], inplace=True)

df.head()

In [ ]:
# Salvar dataset limpo

df.to_csv('../data/processed/valores_arrecadacao_itr_tratados.csv',index=False)

print('Dataset salvo!')

# Etapa 2 — Avaliação da Qualidade dos Dados

Nesta etapa, são verificadas a estrutura do conjunto de dados, a presença de registros duplicados, valores ausentes e possíveis valores discrepantes.

O objetivo é avaliar a consistência da base antes da realização das análises exploratórias.

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.isnull().sum()

In [ ]:

display(df[['valor']].head(10))

nulos = df[df['valor'].isna()]

print("Total de valores nulos:")
print(len(nulos))

print("\nProporção de valores nulos:")
print(len(nulos) / len(df))
nulos.head()

## Tratamento dos valores nulos

Foram identificados valores ausentes na coluna de arrecadação. Para viabilizar as análises e operações de agregação, esses registros foram substituídos por 0.

Essa decisão foi adotada considerando que, no contexto do ITR, existem situações previstas em lei nas quais a arrecadação pode ser igual a zero, como casos de isenção, imunidade tributária e outras condições legais aplicáveis. Além disso, a quantidade de registros com valores ausentes representa uma parcela muito pequena da base de dados, reduzindo o impacto dessa substituição nas análises. Como a base pública utilizada não disponibiliza documentação especificando o significado dos valores ausentes, essa conversão foi tratada como uma premissa metodológica do projeto.

In [ ]:
df['valor'] = df['valor'].fillna(0)
print(df['valor'].isna().sum())

df.info()

In [ ]:
df.describe()

In [ ]:
# Boxplot para verificar outliers

plt.figure(figsize=(10,5))

plt.boxplot(df['valor'])

plt.title('Boxplot da arrecadação do ITR')
plt.ylabel('Valor arrecadado')

plt.show()

# Etapa 3 — Análise Exploratória de Dados (EDA)

# Pergunta 1 — Quais estados arrecadaram mais ITR?

In [ ]:
top_estados = (
    df.groupby('uf_sigla')['valor'].sum().sort_values(ascending=False)
)

top_estados.head(10)



In [ ]:
plt.figure(figsize=(12,6))

top_estados.head(10).plot(kind='bar')

plt.title('Top 10 estados com maior arrecadação de ITR')
plt.xlabel('Estado')
plt.ylabel('Valor arrecadado')
plt.xticks(rotation=45)

plt.savefig("../images/maior_arrecadacao_por_estado.png", dpi=300, bbox_inches="tight")

plt.show()

Os resultados mostram que a arrecadação do ITR está concentrada em determinados estados, com destaque para Mato Grosso do Sul, São Paulo e Mato Grosso.

Esses estados apresentam valores acumulados superiores aos das demais unidades federativas analisadas. Esse comportamento pode estar relacionado a fatores como extensão territorial, quantidade e valor das propriedades rurais e intensidade da atividade agropecuária.

Entretanto, seriam necessárias outras fontes de dados para avaliar a influência individual de cada fator.

# Pergunta 2 — A arrecadação aumentou ao longo dos anos?

In [ ]:
arrecadacao_tempo = (
    df.groupby('ano')['valor'].sum()
)

arrecadacao_tempo

In [ ]:
plt.figure(figsize=(12,6))

plt.plot(
    arrecadacao_tempo.index,
    arrecadacao_tempo.values,
    marker='o'
)

plt.title('Evolução da arrecadação do ITR ao longo dos anos')
plt.xlabel('Ano')
plt.ylabel('Valor arrecadado (R$)')
plt.grid(True)

plt.savefig("../images/evolucao_arrecadacao_itr.png", dpi=300, bbox_inches="tight")

plt.show()

Observa-se uma tendência de crescimento da arrecadação do ITR ao longo do período analisado, com aumento mais acentuado a partir de 2020.

O valor referente a 2026 apresenta uma redução significativa em comparação aos anos anteriores. Como os dados desse ano podem ainda não representar o período completo, o resultado deve ser interpretado com cautela.

# Pergunta 3 — Existe concentração regional de arrecadação?

In [ ]:
regioes = (
    df.groupby('regiao')['valor'].sum().sort_values(ascending=False)
)

regioes

In [ ]:
plt.figure(figsize=(8,8))

plt.pie(regioes,labels=regioes.index,autopct='%1.1f%%')

plt.title('Participação das regiões na arrecadação')

plt.savefig("../images/participacao_regioes.png", dpi=300, bbox_inches="tight")

plt.show()

A análise demonstra uma concentração relevante da arrecadação do ITR na região Centro-Oeste, seguida pelas regiões Sudeste e Sul.

Essa distribuição pode estar associada à extensão das áreas rurais, à quantidade de propriedades e às características econômicas das regiões. No entanto, a confirmação dessas relações exigiria a integração com outras fontes de dados.

# Pergunta 4 — Existe concentração e disparidade na arrecadação entre municípios?

In [ ]:
top_cidades = (
    df.groupby('cidade')['valor'].sum().sort_values(ascending=False)
)

top_cidades.head(10)

In [ ]:
plt.figure(figsize=(12,6))

top_cidades.head(10).plot(kind='bar')

plt.title('Top 10 municípios com maior arrecadação')
plt.xlabel('Município')
plt.ylabel('Valor arrecadado')

plt.xticks(rotation=45)

plt.savefig("../images/maior_arrecadacao_por_cidade.png", dpi=300, bbox_inches="tight")

plt.show()

Os resultados indicam uma forte concentração da arrecadação em um grupo reduzido de municípios.

Maracaju, Campo Grande e Corumbá apresentam valores acumulados significativamente superiores aos da maior parte das cidades analisadas, evidenciando uma distribuição assimétrica da arrecadação municipal.

In [ ]:
plt.figure(figsize=(10,5))

plt.boxplot(top_cidades)

plt.title('Distribuição da arrecadação municipal')
plt.ylabel('Valor arrecadado')

plt.savefig("../images/distribuicao_arrecadacao_municipal.png", dpi=300, bbox_inches="tight")

plt.show()

O boxplot reforça a presença de uma distribuição assimétrica, com grande diferença entre os valores arrecadados pelos municípios.

A presença de pontos distantes da maior concentração dos dados indica que alguns municípios possuem valores consideravelmente superiores aos demais. Esses registros não representam necessariamente erros, mas municípios com comportamento distinto dentro do conjunto analisado.

# Conclusões

A análise identificou uma distribuição concentrada da arrecadação do ITR entre estados, regiões e municípios. A região Centro-Oeste apresentou participação relevante, enquanto alguns estados e municípios concentraram valores significativamente superiores aos demais.

Também foi observada uma tendência de crescimento da arrecadação ao longo do período analisado. Entretanto, o resultado referente a 2026 deve ser interpretado com cautela, pois o ano pode não estar representado integralmente na base.

Os resultados sugerem que características territoriais, fundiárias e econômicas podem estar relacionadas às diferenças observadas. Contudo, a confirmação dessas relações exigiria a integração dos dados de arrecadação com outras informações, como área rural, quantidade de propriedades, produção agropecuária e valor da terra.

Como evolução futura, o projeto pode incluir novas fontes de dados, automatização do processo de tratamento e desenvolvimento de um dashboard interativo.